In [100]:
# This code is part of QCMet.
# 
# (C) Copyright 2024 National Physical Laboratory and National Quantum Computing Centre 
# 
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
# 
#      http://www.apache.org/licenses/LICENSE-2.0
# 
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

import sys
import pathlib
import os
sys.path.insert(0, str(pathlib.Path(os.path.abspath('')).parent.parent.parent.resolve()))

from _helpers.noise_model import *

# Construct the mid-circuit measurement circuit

In [28]:
control = QuantumRegister(1, name="control")
target = QuantumRegister(1, name="target")
mid_measure = ClassicalRegister(1, name="mid")
final_measure = ClassicalRegister(2, name="final")

qc = QuantumCircuit(control, target, mid_measure, final_measure)
qc.h(control)
qc.cx(control, target)
qc.measure(target, mid_measure)
qc.cx(control, target)
qc.measure(control, final_measure[0])
qc.measure(target, final_measure[1])
print(qc)

         ┌───┐             ┌─┐   
control: ┤ H ├──■───────■──┤M├───
         └───┘┌─┴─┐┌─┐┌─┴─┐└╥┘┌─┐
 target: ─────┤ X ├┤M├┤ X ├─╫─┤M├
              └───┘└╥┘└───┘ ║ └╥┘
  mid: 1/═══════════╩═══════╬══╬═
                    0       ║  ║ 
final: 2/═══════════════════╩══╩═
                            0  1 


# Test whether the mid-circuit measurement executes 

In [29]:
try:
    simulator = AerSimulator(noise_model=custom_noise_model())
    result = simulator.run(qc, shots=1000).result()
    counts = result.get_counts()
    # Reverse qiskit little endian qubit order into big endian for better understandability
    counts = {key[::-1]: value for key, value in counts.items()}
    print("There were no errors during the execution.")
    print("The device may support mid-circuit measurements, but you need to inspect the obtained measurement counts.")
    print(f"\nThe measurement counts are {counts}.")
except Exception as e:
    print(e)
    print("Exception caught! The device does not support mid-circuit measurements.")
    

There were no errors during the execution.
The device may support mid-circuit measurements, but you need to inspect the obtained measurement counts.

The measurement counts are {'0 01': 2, '1 01': 6, '1 00': 16, '0 11': 10, '1 10': 472, '0 10': 2, '0 00': 492}.


# Inspect the result
The three measurements of the circuit should read out the following two sets of states with equal probability: {$\ket{0},\ket{00}$} and {$\ket{1}, \ket{10}$}. 

Small contributions of other states can be due to the noise present in the device. 
Large contributions of other states means that the mid-circuit measurement undesirably affects the first and/or the second qubit.

## For AWS Braket
The currently installed version of AWS Braket does not support mid-circuit measurements because there is no `measurement()` command for ciruits, and on the hardware, the measurements always take place automatically at the end of circuit. 

This can be seen with the following cell, which throws an error.

In [ ]:
from braket.circuits import Circuit

braket_circuit = Circuit()
braket_circuit.measure()

AttributeError: 'Circuit' object has no attribute 'measure'